# Day 7 · Exercise 3: Summarize a Single Chunk

**What you'll build:** `summarize_chunk(chunk: str, model: str) -> str` — a function that sends one text chunk to a local Ollama model with a focused system prompt and returns a plain-prose summary string.

**Why it matters:** A well-scoped summarization function is the atomic unit of every long-document pipeline — once it works reliably in isolation, every larger pipeline (chunking loops, reduce steps, batch jobs) can call it without re-implementing the model interaction logic.

## Your Implementation

In [ ]:
import ollama

SUMMARIZE_SYSTEM_PROMPT = (
    # ── YOUR SYSTEM PROMPT HERE ───────────────────────────────
    ""
    # ───────────────────────────────────────────────────────────
)


def summarize_chunk(chunk: str, model: str) -> str:
    """Summarize a single text chunk using a local Ollama model.

    Args:
        chunk: A string containing the text to summarize.
               Should already fit within the model's context window.
        model: The name of the Ollama model to use (e.g. "llama3.2").

    Returns:
        A plain-prose summary string (2–3 sentences). The summary
        contains only information present in the chunk — no elaboration.

    Example:
        >>> summary = summarize_chunk(
        ...     "The Eiffel Tower was built in 1889 for the World's Fair.",
        ...     model="llama3.2",
        ... )
        >>> isinstance(summary, str) and len(summary) > 0
        True
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

_SAMPLE_CHUNK = (
    "The city of Brasília was designed by urban planner Lúcio Costa and "
    "architect Oscar Niemeyer and was inaugurated as Brazil's capital in 1960. "
    "Its layout is often compared to the shape of an airplane or a bird in "
    "flight when viewed from above. The city was inscribed as a UNESCO World "
    "Heritage Site in 1987 in recognition of its modernist architecture and "
    "innovative city planning."
)

_SHORT_CHUNK = "The Eiffel Tower was built in 1889 as the entrance arch for the World's Fair."

_MODEL = "llama3.2"


def _run_checks():
    score, total = 0, 4

    # Check 1: function exists and is callable
    try:
        assert callable(summarize_chunk), 'summarize_chunk is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: returns a non-empty string for a standard chunk
    result1 = None
    try:
        result1 = summarize_chunk(_SAMPLE_CHUNK, model=_MODEL)
        assert isinstance(result1, str), (
            f'expected str, got {type(result1).__name__}'
        )
        assert len(result1.strip()) > 0, 'returned an empty string'
        print(f'{_PASS} Check 2/{total}: returns a non-empty string ({len(result1)} chars)')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')

    # Check 3: output is not a verbatim echo of the input
    # (guards against a missing or broken system prompt)
    try:
        if result1 is None:
            raise AssertionError('skipped — Check 2 did not produce a result')
        # A proper summary will be noticeably shorter than the chunk,
        # and will not start with the exact same opening words.
        chunk_start = _SAMPLE_CHUNK[:60].lower()
        result_start = result1[:60].lower()
        assert result_start != chunk_start, (
            'output appears to be a verbatim echo of the input — '
            'check that SUMMARIZE_SYSTEM_PROMPT is non-empty and passed correctly'
        )
        print(f'{_PASS} Check 3/{total}: output is a summary, not a verbatim echo')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: short one-sentence chunk still produces a non-empty string
    try:
        short_result = summarize_chunk(_SHORT_CHUNK, model=_MODEL)
        assert isinstance(short_result, str), (
            f'expected str, got {type(short_result).__name__}'
        )
        assert len(short_result.strip()) > 0, 'returned an empty string for a short chunk'
        print(f'{_PASS} Check 4/{total}: short one-sentence chunk returns a non-empty summary')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {total}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

Right now `summarize_chunk` calls the model but gives you no visibility into how long it took. Time the `ollama.chat()` call and print a DEBUG-level log line that shows the model name, chunk length (in characters), and elapsed time in milliseconds.

This foreshadows Day 8, where you will add structured logging throughout the pipeline so every model call is observable — latency, token counts, and error context included.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import ollama

SUMMARIZE_SYSTEM_PROMPT = (
    "You are a precise summarization assistant. "
    "The user will send you a passage of text — possibly an excerpt from a "
    "larger document. Summarize it in 2–3 sentences of plain prose. "
    "Do not add headings, bullet points, or information that is not present "
    "in the passage. Use only what is written."
)


def summarize_chunk(chunk: str, model: str) -> str:
    """Summarize a single text chunk using a local Ollama model."""
    messages = [
        {"role": "system", "content": SUMMARIZE_SYSTEM_PROMPT},
        {"role": "user",   "content": chunk},
    ]
    response = ollama.chat(model=model, messages=messages)
    return response["message"]["content"]
```

**Why this works:** The system prompt does four things at once — it assigns a role ("precise summarization assistant"), constrains the output length (2–3 sentences), locks the format (plain prose, no headings or bullet points), and prohibits the model from adding information not in the passage. Without that last constraint, a language model will happily fill gaps with training knowledge, corrupting the summary when the chunk is an excerpt of a larger document. The function itself is intentionally minimal: build the messages list, call `ollama.chat`, index into `response["message"]["content"]`, and return the string — one job, four lines.
</details>